# hur6 + amplitude encoding puro

Questo notebook fa un run di ablation con `hur6` e amplitude encoding senza la rotazione aggiuntiva `RY(norm_angle)` su `wire=0`.

Nel codice della suite, `e3` = `AmplitudeEmbedding(normalize=True)` + `RY(norm_angle)`. Qui registriamo temporaneamente un encoding `e3_plain` che fa solo `AmplitudeEmbedding(normalize=True)`.

In [1]:
import os
from pathlib import Path

import pennylane as qml
import torch

from suitev2.config import ENCODING_REGISTRY, populate_registries
from suitev2.model import FashionQCNN, WIRES_8, WIRES_4, OUTPUT_WIRES, POOL_PAIRS_LAYER1, POOL_PAIRS_LAYER2
from suitev2.ansatz import convolution_layer_on_wires, pooling_layer
import suitev2.train as train_mod
from suitev2.run_suite import ensure_summary_header, append_summary_row

populate_registries()
print("Encoding disponibili:", list(ENCODING_REGISTRY.keys()))

Encoding disponibili: ['e1', 'e3', 'custom']


## Configurazione run

`TRAIN_PER_CLASS = None` usa la dimensione completa della suite. Per un controllo veloce, metti ad esempio `TRAIN_PER_CLASS = 50` e `EPOCHS = 1`.

In [2]:
ANSATZ = "hur6"
ENCODING = "e3_plain"
SEED = 42

EPOCHS = 10
TRAIN_PER_CLASS = None

RESULTS_DIR = Path("suitev2") / "results_hur6_e3_plain"
RUN_DIR = RESULTS_DIR / f"{ANSATZ}_{ENCODING}_seed{SEED}"
SUMMARY_PATH = RESULTS_DIR / "summary.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("run_dir:", RUN_DIR)
print("summary:", SUMMARY_PATH)

run_dir: suitev2\results_hur6_e3_plain\hur6_e3_plain_seed42
summary: suitev2\results_hur6_e3_plain\summary.csv


## Encoding amplitude puro

Questa funzione non riceve e non usa `norm_angle`, quindi non applica la `RY` aggiuntiva di luminosita/norma.

In [3]:
def encoding_e3_plain(x_flat, n_qubits=8):
    qml.AmplitudeEmbedding(features=x_flat, wires=range(n_qubits), normalize=True)


ENCODING_REGISTRY["e3_plain"] = (encoding_e3_plain, 8, False)
print("Registrato e3_plain:", ENCODING_REGISTRY["e3_plain"])

Registrato e3_plain: (<function encoding_e3_plain at 0x00000236CA2DF9C0>, 8, False)


## Modello temporaneo per `e3_plain`

La classe originale `FashionQCNN` gestisce esplicitamente `e3`, `e1` e `custom`. Questa sottoclasse aggiunge solo il ramo `e3_plain`, lasciando invariati ansatz, pooling, loss, optimizer e data loader.

In [4]:
class FashionQCNNPlainAmplitude(FashionQCNN):
    def _build_qnode(self):
        if self.encoding_name != "e3_plain":
            return super()._build_qnode()

        conv_fn = self.conv_fn
        enc_fn = self.enc_fn
        pool_fn = self.pool_fn
        use_param_pool = self.use_param_pool
        use_final_classifier = self.use_final_classifier

        def apply_pool(theta_pool, pool_pairs):
            if use_param_pool:
                pooling_layer(pool_fn, theta_pool, pool_pairs)

        def apply_final(theta_final):
            if use_final_classifier:
                conv_fn(theta_final, OUTPUT_WIRES)

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode_e3_plain(x_flat, theta_c1, theta_p1, theta_c2, theta_p2, theta_final):
            enc_fn(x_flat, n_qubits=8)

            convolution_layer_on_wires(conv_fn, theta_c1, WIRES_8)
            apply_pool(theta_p1, POOL_PAIRS_LAYER1)

            convolution_layer_on_wires(conv_fn, theta_c2, WIRES_4)
            apply_pool(theta_p2, POOL_PAIRS_LAYER2)
            apply_final(theta_final)

            return qml.probs(wires=OUTPUT_WIRES)

        return qnode_e3_plain

    def forward(self, x_flat, quad_means_8=None, gA4=None):
        if self.encoding_name != "e3_plain":
            return super().forward(x_flat, quad_means_8=quad_means_8, gA4=gA4)

        probs = self.qnode(
            x_flat,
            self.theta_conv1, self.theta_pool1,
            self.theta_conv2, self.theta_pool2,
            self.theta_final if self.theta_final is not None else x_flat.new_empty(0),
        )
        return probs.float()


train_mod.FashionQCNN = FashionQCNNPlainAmplitude

sanity_model = FashionQCNNPlainAmplitude(ANSATZ, ENCODING)
print("n_qubits:", sanity_model.n_qubits)
print("n_trainable_params:", sanity_model.n_trainable_params())

n_qubits: 8
n_trainable_params: 16


## Lancia il run

In [7]:
ensure_summary_header(str(SUMMARY_PATH))

result = train_mod.train_one_run(
    ANSATZ,
    ENCODING,
    59,
    str(RUN_DIR),
    epochs=EPOCHS,
    train_per_class=TRAIN_PER_CLASS,
    log_fn=print,
)

append_summary_row(str(SUMMARY_PATH), result)
result

  epoch 01 | train_loss=1.1409 acc=0.6193 | val_loss=1.0170 acc=0.6420 | gnorm=4.516e-01 | 127.9s
  epoch 02 | train_loss=1.0395 acc=0.6362 | val_loss=0.9387 acc=0.6475 | gnorm=3.395e-01 | 94.6s
  epoch 03 | train_loss=1.0014 acc=0.6628 | val_loss=0.9191 acc=0.6840 | gnorm=2.922e-01 | 131.7s
  epoch 04 | train_loss=0.9726 acc=0.7401 | val_loss=0.8775 acc=0.7765 | gnorm=2.663e-01 | 129.3s
  epoch 05 | train_loss=0.9468 acc=0.7912 | val_loss=0.8584 acc=0.8090 | gnorm=2.539e-01 | 128.5s
  epoch 06 | train_loss=0.9352 acc=0.8105 | val_loss=0.8511 acc=0.8125 | gnorm=2.440e-01 | 130.2s
  epoch 07 | train_loss=0.9234 acc=0.8155 | val_loss=0.8336 acc=0.8145 | gnorm=2.529e-01 | 143.9s
  epoch 08 | train_loss=0.9114 acc=0.8128 | val_loss=0.8245 acc=0.8130 | gnorm=2.498e-01 | 144.1s


KeyboardInterrupt: 

## Risultato sintetico

In [6]:
print(f"test_acc  = {result['test_acc']:.4f}")
print(f"test_loss = {result['test_loss']:.4f}")
print(f"best_epoch = {result['best_epoch']}")
print("confusion_matrix:")
for row in result["confusion_matrix"]:
    print(row)

print("\nFile scritti:")
print("-", RUN_DIR / "config.json")
print("-", RUN_DIR / "metrics.csv")
print("-", RUN_DIR / "test.json")
print("-", SUMMARY_PATH)

test_acc  = 0.8010
test_loss = 0.8201
best_epoch = 10
confusion_matrix:
[395, 93, 3, 9]
[22, 477, 1, 0]
[2, 2, 362, 134]
[125, 2, 5, 368]

File scritti:
- suitev2\results_hur6_e3_plain\hur6_e3_plain_seed42\config.json
- suitev2\results_hur6_e3_plain\hur6_e3_plain_seed42\metrics.csv
- suitev2\results_hur6_e3_plain\hur6_e3_plain_seed42\test.json
- suitev2\results_hur6_e3_plain\summary.csv
